## ✔️ Accelerator Checking

In [1]:
import torch
print(torch.cuda.is_available())        # Should print: True
print(torch.cuda.get_device_name(0))

True
Tesla T4


## ✔️ Checking Folder Structure

In [2]:
# import os
# base = "/kaggle/input/datasets/omkarmanohardalvi/lungs-disease-dataset-4-types"
# return # remove it if you want to check structure
# for root, dirs, files in os.walk(base):
#     depth = root.replace(base, '').count(os.sep)
#     if depth <= 2:
#         print("  " * depth + os.path.basename(root) + "/",
#               f"({len(files)} files)" if files else "")

__________
# 🫁 Lung Disease CNN Classifier
### Multimodal Architecture — CNN Branch
**Dataset:** Lungs Disease Dataset (4 types) — Kaggle  
**Models:** ResNet50 · VGG16 · ResNet18  
**Architecture:** Backbone → DepthwiseSeparableConv → GCSA → Feature Vector (512-d)   
**Optimizations:** Transfer Learning · Layer Freezing · Mixed Precision (AMP)
· Dataset Caching · Prefetching · Parallel Workers   
**Metrics:** Accuracy · Recall · F1 Score · Specificity


## ⚙️ Cell 1 — Configuration
> **Change settings here only. No need to touch any other cell.**  
> - `QUICK_TEST = True` → 1/4 data, 5 epochs (verify everything works)
> - `QUICK_TEST = False` → full training
> - `FREEZE_EPOCHS` → epochs to keep backbone frozen before full fine-tuning

In [3]:
import torch
 
# ── Dataset ───────────────────────────────────────────────
DATA_ROOT = "/kaggle/input/datasets/omkarmanohardalvi/lungs-disease-dataset-4-types/Lung Disease Dataset"
IMG_SIZE  = 224
 
# ── Quick test mode (True = 1/4 data, fewer epochs) ───────
QUICK_TEST = False                    # ← Set False for full training
EPOCHS      = 5  if QUICK_TEST else 30
SUBSET_FRAC = 0.25 if QUICK_TEST else 1.0
 
# ── Training ──────────────────────────────────────────────
BATCH_SIZE    = 64
LR            = 1e-4
FEATURE_DIM   = 512
DROPOUT       = 0.4
NUM_WORKERS   = 4
PREFETCH      = 2
FREEZE_EPOCHS = 2

# ── Backbones ───────────────────────────────────────────────
BACKBONES = ["resnet50", "resnet18", "vgg16"]

# ── Device & AMP ────────────────────────────────────────────
DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()
 
# ── Classes (exact Kaggle folder names) ───────────────────
CLASS_NAMES  = ["Bacterial Pneumonia", "Corona Virus Disease",
                "Normal", "Tuberculosis", "Viral Pneumonia"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
NUM_CLASSES  = len(CLASS_NAMES)
 
print(f"Device       : {DEVICE}")
print(f"AMP enabled  : {USE_AMP}")
print(f"Quick Test   : {QUICK_TEST}  (subset={SUBSET_FRAC}, epochs={EPOCHS})")
print(f"Batch size   : {BATCH_SIZE}")
print(f"Workers      : {NUM_WORKERS}  prefetch={PREFETCH}")
print(f"Freeze epochs: {FREEZE_EPOCHS}")

Device       : cuda
AMP enabled  : True
Quick Test   : False  (subset=1.0, epochs=30)
Batch size   : 64
Workers      : 4  prefetch=2
Freeze epochs: 2


## 📦 Cell 2 — Imports & Cached Dataset
**Key optimizations:**
- All images pre-loaded into RAM → zero disk reads after first load
- `num_workers=4` parallel batch loading
- `prefetch_factor=2` keeps GPU fed ahead of time
- `persistent_workers=True` avoids worker restart overhead each epoch

In [4]:
import os, time, copy
from pathlib import Path
 
import numpy as np
import matplotlib
matplotlib.use("Agg")                  # non-interactive backend for Kaggle
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
 
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset, Subset
from torch.amp import GradScaler, autocast # "torch.cuda.amp" throws a future warning since these APIs are depricated
from torchvision import models, transforms
from sklearn.metrics import roc_curve, auc
from PIL import Image
 
 
class CachedLungDataset(Dataset):
    """All images pre-loaded into RAM. Augmentation applied on-the-fly."""
    EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
 
    def __init__(self, root, split="train", img_size=224):
        self.transform = self._get_transforms(split, img_size)
        split_dir      = Path(root) / split
        raw_paths      = []
 
        for cls in CLASS_NAMES:
            d = split_dir / cls
            if not d.exists():
                print(f"  [!] Missing: {d}")
                continue
            for f in d.iterdir():
                if f.suffix.lower() in self.EXT:
                    raw_paths.append((str(f), CLASS_TO_IDX[cls]))
 
        print(f"  [{split.upper():5s}] Loading {len(raw_paths):,} images into RAM...",
              end="", flush=True)
        t0 = time.time()
        self.samples = []
        for path, label in raw_paths:
            img = Image.open(path).convert("L")
            img.load()                          # force full decode now
            self.samples.append((img, label))
        print(f" done in {time.time()-t0:.1f}s")
 
    @staticmethod
    def _get_transforms(split, img_size):
        norm = transforms.Normalize([0.485, 0.456, 0.406],
                                    [0.229, 0.224, 0.225])
        if split == "train":
            return transforms.Compose([
                transforms.Grayscale(3),
                transforms.Resize((img_size + 32, img_size + 32)),
                transforms.RandomCrop(img_size),
                transforms.RandomHorizontalFlip(),
                transforms.RandomRotation(10),
                transforms.ColorJitter(brightness=0.2, contrast=0.2),
                transforms.ToTensor(), norm,
            ])
        return transforms.Compose([
            transforms.Grayscale(3),
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(), norm,
        ])
 
    def __len__(self):
        return len(self.samples)
 
    def __getitem__(self, i):
        img, label = self.samples[i]
        return self.transform(img), label
 
 
def build_dataloaders(subset_frac=1.0, img_size=IMG_SIZE):
    train_ds = CachedLungDataset(DATA_ROOT, "train", img_size)
    val_ds   = CachedLungDataset(DATA_ROOT, "val",   img_size)
 
    if subset_frac < 1.0:
        n_tr     = int(len(train_ds) * subset_frac)
        n_vl     = int(len(val_ds)   * subset_frac)
        train_ds = Subset(train_ds, torch.randperm(len(train_ds))[:n_tr].tolist())
        val_ds   = Subset(val_ds,   torch.randperm(len(val_ds))[:n_vl].tolist())
        print(f"  [SUBSET] train={n_tr}, val={n_vl}")
 
    pin = torch.cuda.is_available()
    kw  = dict(num_workers=NUM_WORKERS, pin_memory=pin,
               prefetch_factor=PREFETCH, persistent_workers=(NUM_WORKERS > 0))
 
    return (DataLoader(train_ds, BATCH_SIZE, shuffle=True,  **kw),
            DataLoader(val_ds,   BATCH_SIZE, shuffle=False, **kw))
 
 
def build_test_loader(img_size=IMG_SIZE):
    """Test loader — always full dataset, never subset."""
    test_ds = CachedLungDataset(DATA_ROOT, "test", img_size)
    pin     = torch.cuda.is_available()
    return DataLoader(test_ds, BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=pin,
                      prefetch_factor=PREFETCH,
                      persistent_workers=(NUM_WORKERS > 0))
 
 
print("Dataset class defined ✓")

Dataset class defined ✓


## 🧠 Cell 3 — Model Architecture
**DepthwiseSeparableConv → GCSA (Global Context Self-Attention)**  
Supports: ResNet50 · VGG16 · ResNet18   
Backbone frozen for first `FREEZE_EPOCHS`, then fully unfrozen at 10× lower LR.

In [5]:
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, padding=1):
        super().__init__()
        self.dw   = nn.Conv2d(in_ch, in_ch, kernel_size, padding=padding,
                              groups=in_ch, bias=False)
        self.pw   = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.bn   = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
 
    def forward(self, x):
        return self.relu(self.bn(self.pw(self.dw(x))))
 
 
class GCSA(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.query   = nn.Conv2d(channels, 1, 1)
        mid          = max(channels // reduction, 1)
        self.ch_attn = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(channels, mid), nn.ReLU(inplace=True),
            nn.Linear(mid, channels), nn.Sigmoid()
        )
        self.value = nn.Conv2d(channels, channels, 1)
        self.gamma = nn.Parameter(torch.zeros(1))
        self.bn    = nn.BatchNorm2d(channels)
 
    def forward(self, x):
        B, C, H, W = x.shape
        attn = F.softmax(self.query(x).view(B, 1, -1), dim=-1)
        val  = self.value(x).view(B, C, -1)
        ctx  = torch.bmm(val, attn.permute(0, 2, 1)).view(B, C, 1, 1)
        ch   = self.ch_attn(x).view(B, C, 1, 1)
        return self.bn(x + self.gamma * ctx * ch)
 
 
class LungCNNClassifier(nn.Module):
    def __init__(self, backbone, num_classes, feature_dim=512,
                 pretrained=True, dropout=0.4):
        super().__init__()
        self.backbone_name = backbone
        feat, ds, gcsa, gap, proj = self._build(backbone, feature_dim, pretrained)
        self.features = feat
        self.ds       = ds
        self.gcsa     = gcsa
        self.gap      = gap
        self.proj     = proj
        self.head     = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feature_dim, num_classes)
        )
 
    def _build(self, name, feature_dim, pretrained):
        if name == "resnet50":
            w  = models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
            bb = models.resnet50(weights=w)
            feat  = nn.Sequential(bb.conv1, bb.bn1, bb.relu, bb.maxpool,
                                  bb.layer1, bb.layer2, bb.layer3, bb.layer4)
            in_ch = 2048
        elif name == "resnet18":
            w  = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
            bb = models.resnet18(weights=w)
            feat  = nn.Sequential(bb.conv1, bb.bn1, bb.relu, bb.maxpool,
                                  bb.layer1, bb.layer2, bb.layer3, bb.layer4)
            in_ch = 512
        elif name == "vgg16":
            w  = models.VGG16_Weights.IMAGENET1K_V1 if pretrained else None
            bb = models.vgg16(weights=w)
            feat  = bb.features
            in_ch = 512
        else:
            raise ValueError(f"Unknown backbone: {name}")
 
        return (feat,
                DepthwiseSeparableConv(in_ch, 512),
                GCSA(512),
                nn.AdaptiveAvgPool2d(1),
                nn.Linear(512, feature_dim))
 
    def freeze_backbone(self):
        for p in self.features.parameters():
            p.requires_grad = False
 
    def unfreeze_backbone(self):
        for p in self.features.parameters():
            p.requires_grad = True
 
    def extract_features(self, x):
        x = self.features(x)
        x = self.ds(x)
        x = self.gcsa(x)
        x = self.gap(x).flatten(1)
        return self.proj(x)             # (B, 512) — for Feature Fusion
 
    def forward(self, x):
        return self.head(self.extract_features(x))
 
 
print("Model architecture defined ✓")

Model architecture defined ✓


## 📊 Cell 4 — Metrics & Training Functions
**Metrics computed from the confusion matrix:**
- **Accuracy** — overall correct predictions
- **Recall (Sensitivity)** — macro-average true positive rate per class
- **F1 Score** — macro-average harmonic mean of precision and recall
- **Specificity** — macro-average true negative rate per class
 
All metrics are computed on both val and test sets.
Test set evaluation is integrated directly inside `run_training()`
so results appear immediately after each model finishes.

In [6]:
def compute_metrics(y_true, y_pred, num_classes):
    """
    Compute Accuracy, Macro Recall, Macro F1, Macro Specificity
    from flat numpy arrays of true and predicted labels.
    Returns a dict with float values.
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
 
    # ── Confusion matrix (C[i,j] = predicted j, actual i) ─
    C = np.zeros((num_classes, num_classes), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        C[t, p] += 1
 
    accuracy = np.trace(C) / C.sum()
 
    recalls, precisions, specificities, f1s = [], [], [], []
    for i in range(num_classes):
        tp = C[i, i]
        fn = C[i, :].sum() - tp          # actual i, predicted not-i
        fp = C[:, i].sum() - tp          # predicted i, actual not-i
        tn = C.sum() - tp - fn - fp
 
        recall      = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        precision   = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        f1          = (2 * precision * recall / (precision + recall)
                       if (precision + recall) > 0 else 0.0)
 
        recalls.append(recall)
        precisions.append(precision)
        specificities.append(specificity)
        f1s.append(f1)
 
    return {
        "accuracy"   : float(accuracy),
        "recall"     : float(np.mean(recalls)),
        "f1"         : float(np.mean(f1s)),
        "specificity": float(np.mean(specificities)),
        "confusion"  : C,
        "per_class"  : {
            CLASS_NAMES[i]: {
                "recall"     : recalls[i],
                "precision"  : precisions[i],
                "f1"         : f1s[i],
                "specificity": specificities[i],
            } for i in range(num_classes)
        }
    }
 
 
def train_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    loss_sum, y_true, y_pred = 0., [], []
 
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
 
        with autocast("cuda", enabled=USE_AMP): # need to add "cuda"(device_type) here
            out  = model(imgs)
            loss = criterion(out, labels)
 
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
 
        loss_sum += loss.item() * len(imgs)
        y_true.extend(labels.cpu().tolist())
        y_pred.extend(out.argmax(1).cpu().tolist())
 
    metrics = compute_metrics(y_true, y_pred, NUM_CLASSES)
    return loss_sum / len(y_true), metrics
 
 
@torch.no_grad()
def evaluate_loader(model, loader, criterion):
    """
    Returns loss, metrics dict, all true labels, all predicted probs.
    Probs are needed for ROC curves.
    """
    model.eval()
    loss_sum       = 0.
    y_true, y_pred = [], []
    y_probs        = []          # softmax probabilities for ROC
 
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast("cuda", enabled=USE_AMP):
            out  = model(imgs)
            loss = criterion(out, labels)
 
        probs = torch.softmax(out, dim=1)
        loss_sum += loss.item() * len(imgs)
        y_true.extend(labels.cpu().tolist())
        y_pred.extend(out.argmax(1).cpu().tolist())
        y_probs.append(probs.cpu())
 
    y_probs  = torch.cat(y_probs, dim=0).numpy()   # (N, C)
    metrics  = compute_metrics(y_true, y_pred, NUM_CLASSES)
    return loss_sum / len(y_true), metrics, y_true, y_probs
 
 
def run_training(backbone_name, test_loader):
    """
    Train one backbone, evaluate on val each epoch,
    then run final evaluation on test set immediately after training.
    Returns history and full test metrics for plotting.
    """
    print(f"\n{'='*65}")
    print(f"  Training : {backbone_name.upper()}")
    print(f"  Mode     : {'QUICK TEST (1/4 data)' if QUICK_TEST else 'FULL TRAINING'}")
    print(f"  AMP      : {USE_AMP}   Freeze epochs: {FREEZE_EPOCHS}")
    print(f"{'='*65}")
 
    train_loader, val_loader = build_dataloaders(SUBSET_FRAC)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
 
    model    = LungCNNClassifier(backbone_name, NUM_CLASSES, FEATURE_DIM,
                                 pretrained=True, dropout=DROPOUT).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"\n  Parameters: {n_params:.2f}M")
 
    model.freeze_backbone()
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
    print(f"  Trainable (frozen phase): {trainable:.2f}M\n")
 
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad,
                                   model.parameters()), lr=LR, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    scaler    = GradScaler("cuda", enabled=USE_AMP) # need to put "cuda"(device_type) here.
    best_val_acc, best_state = 0.0, None
    history = {k: [] for k in ["train_loss", "val_loss",
                                "train_acc",  "val_acc",
                                "train_f1",   "val_f1"]}
 
    print(f"  {'Ep':>3} {'Phase':<10} {'Tr Loss':>8} {'Tr Acc':>7} {'Tr F1':>6} "
          f"{'Val Loss':>9} {'Val Acc':>8} {'Val F1':>7} {'Time':>6}")
    print("  " + "─" * 68)
 
    for epoch in range(1, EPOCHS + 1):
 
        if epoch == FREEZE_EPOCHS + 1:
            model.unfreeze_backbone()
            optimizer = optim.AdamW(model.parameters(),
                                    lr=LR * 0.1, weight_decay=1e-4)
            scheduler = CosineAnnealingLR(optimizer,
                                          T_max=EPOCHS - FREEZE_EPOCHS,
                                          eta_min=1e-6)
            trainable = sum(p.numel() for p in model.parameters()
                            if p.requires_grad) / 1e6
            print(f"  >>> Backbone unfrozen — trainable: {trainable:.2f}M, "
                  f"LR → {LR*0.1:.1e}")
 
        phase = "frozen" if epoch <= FREEZE_EPOCHS else "finetune"
        t0    = time.time()
 
        tr_loss, tr_met = train_epoch(model, train_loader, criterion, optimizer, scaler)
        vl_loss, vl_met, _, _ = evaluate_loader(model, val_loader, criterion)
        scheduler.step()
 
        history["train_loss"].append(tr_loss)
        history["val_loss"].append(vl_loss)
        history["train_acc"].append(tr_met["accuracy"])
        history["val_acc"].append(vl_met["accuracy"])
        history["train_f1"].append(tr_met["f1"])
        history["val_f1"].append(vl_met["f1"])
 
        flag = " ✓" if vl_met["accuracy"] > best_val_acc else ""
        print(f"  {epoch:3d} {phase:<10} {tr_loss:8.4f} {tr_met['accuracy']:7.4f} "
              f"{tr_met['f1']:6.4f} {vl_loss:9.4f} {vl_met['accuracy']:8.4f} "
              f"{vl_met['f1']:7.4f} {time.time()-t0:5.1f}s{flag}")
 
        if vl_met["accuracy"] > best_val_acc:
            best_val_acc = vl_met["accuracy"]
            best_state   = copy.deepcopy(model.state_dict())
 
    # ── Load best weights & evaluate on test set ──────────
    model.load_state_dict(best_state)
    te_loss, te_met, te_true, te_probs = evaluate_loader(
        model, test_loader, criterion)
 
    # ── Save checkpoint ────────────────────────────────────
    save_path = f"/kaggle/working/{backbone_name}_best.pth"
    torch.save({
        "backbone"   : backbone_name,
        "state_dict" : best_state,
        "val_acc"    : best_val_acc,
        "test_metrics": te_met,
        "class_names": CLASS_NAMES,
        "feature_dim": FEATURE_DIM,
        "quick_test" : QUICK_TEST,
    }, save_path)
 
    # ── Print per-model summary ────────────────────────────
    print(f"\n  {'─'*55}")
    print(f"  {'Metric':<14} {'Val (best epoch)':>18}  {'Test Set':>10}")
    print(f"  {'─'*55}")
    print(f"  {'Accuracy':<14} {best_val_acc*100:>17.2f}%  {te_met['accuracy']*100:>9.2f}%")
    print(f"  {'Recall':<14} {'—':>18}  {te_met['recall']*100:>9.2f}%")
    print(f"  {'F1 Score':<14} {'—':>18}  {te_met['f1']*100:>9.2f}%")
    print(f"  {'Specificity':<14} {'—':>18}  {te_met['specificity']*100:>9.2f}%")
    print(f"  {'─'*55}")
    print(f"  ✓ Saved → {save_path}\n")
 
    return history, te_met, te_true, te_probs, best_val_acc
 
 
print("Metrics & training functions defined ✓")

Metrics & training functions defined ✓


## 🚀 Cell 5 — Run All 3 Models Sequentially
Test set evaluation runs **immediately after each model** finishes training.
Each model saves to its own `.pth` file — nothing gets overwritten.

 
| File | Description |
|---|---|
| `resnet50_best.pth` | Best ResNet50 checkpoint |
| `resnet18_best.pth` | Best ResNet18 checkpoint |
| `vgg16_best.pth` | Best VGG16 checkpoint |

In [7]:
# Build test loader once — shared across all models
test_loader = build_test_loader()
 
all_results = {}   # backbone → {history, te_met, te_true, te_probs, val_acc}
 
for backbone in BACKBONES:
    history, te_met, te_true, te_probs, val_acc = run_training(backbone, test_loader)
    all_results[backbone] = {
        "history" : history,
        "te_met"  : te_met,
        "te_true" : te_true,
        "te_probs": te_probs,
        "val_acc" : val_acc,
    }
 
# ── Final comparison table ─────────────────────────────────
print(f"\n{'='*72}")
print(f"  FINAL RESULTS SUMMARY")
print(f"{'='*72}")
print(f"  {'Model':<12} {'Val Acc':>8} {'Test Acc':>10} "
      f"{'Recall':>8} {'F1':>8} {'Specificity':>13}")
print(f"  {'─'*68}")
for name, r in all_results.items():
    m = r["te_met"]
    print(f"  {name.upper():<12} {r['val_acc']*100:>7.2f}%"
          f" {m['accuracy']*100:>9.2f}%"
          f" {m['recall']*100:>7.2f}%"
          f" {m['f1']*100:>7.2f}%"
          f" {m['specificity']*100:>12.2f}%")
print(f"{'='*72}")
print("Note: Recall, F1, Specificity are macro-averaged across all 5 classes.")
print("      Test set was never seen during training or model selection.")

  [TEST ] Loading 2,025 images into RAM... done in 30.2s

  Training : RESNET50
  Mode     : FULL TRAINING
  AMP      : True   Freeze epochs: 2
  [TRAIN] Loading 6,054 images into RAM... done in 91.9s
  [VAL  ] Loading 2,016 images into RAM... done in 30.4s
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 166MB/s]



  Parameters: 25.14M
  Trainable (frozen phase): 1.63M

   Ep Phase       Tr Loss  Tr Acc  Tr F1  Val Loss  Val Acc  Val F1   Time
  ────────────────────────────────────────────────────────────────────
    1 frozen       0.9994  0.6977 0.6875    0.8639   0.7594  0.7527  54.3s ✓
    2 frozen       0.7965  0.8008 0.7978    0.7980   0.8001  0.7954  50.1s ✓
  >>> Backbone unfrozen — trainable: 25.14M, LR → 1.0e-05
    3 finetune     0.7417  0.8320 0.8302    0.7459   0.8214  0.8167  53.7s ✓
    4 finetune     0.6964  0.8561 0.8545    0.7201   0.8358  0.8304  54.6s ✓
    5 finetune     0.6652  0.8731 0.8719    0.7023   0.8462  0.8390  54.4s ✓
    6 finetune     0.6420  0.8821 0.8809    0.6806   0.8586  0.8529  54.4s ✓
    7 finetune     0.6195  0.8999 0.8990    0.6722   0.8681  0.8640  54.3s ✓
    8 finetune     0.6073  0.9062 0.9053    0.6678   0.8676  0.8632  53.7s
    9 finetune     0.5959  0.9123 0.9117    0.6617   0.8755  0.8725  55.1s ✓
   10 finetune     0.5870  0.9174 0.9167    0.65

100%|██████████| 44.7M/44.7M [00:00<00:00, 170MB/s]



  Parameters: 12.01M
  Trainable (frozen phase): 0.83M

   Ep Phase       Tr Loss  Tr Acc  Tr F1  Val Loss  Val Acc  Val F1   Time
  ────────────────────────────────────────────────────────────────────
    1 frozen       1.1306  0.6371 0.6261    0.9301   0.7386  0.7318  53.8s ✓
    2 frozen       0.8738  0.7570 0.7517    0.8330   0.7912  0.7871  50.1s ✓
  >>> Backbone unfrozen — trainable: 12.01M, LR → 1.0e-05
    3 finetune     0.8019  0.7977 0.7935    0.7725   0.8219  0.8163  51.2s ✓
    4 finetune     0.7439  0.8294 0.8269    0.7470   0.8353  0.8313  51.5s ✓
    5 finetune     0.7171  0.8388 0.8364    0.7344   0.8423  0.8374  50.9s ✓
    6 finetune     0.6957  0.8477 0.8454    0.7221   0.8467  0.8431  50.7s ✓
    7 finetune     0.6782  0.8642 0.8625    0.7065   0.8507  0.8480  51.4s ✓
    8 finetune     0.6629  0.8748 0.8735    0.7032   0.8586  0.8553  51.0s ✓
    9 finetune     0.6530  0.8786 0.8774    0.7087   0.8586  0.8544  51.1s
   10 finetune     0.6421  0.8811 0.8797    0.70

100%|██████████| 528M/528M [00:03<00:00, 167MB/s]



  Parameters: 15.55M
  Trainable (frozen phase): 0.83M

   Ep Phase       Tr Loss  Tr Acc  Tr F1  Val Loss  Val Acc  Val F1   Time
  ────────────────────────────────────────────────────────────────────
    1 frozen       1.1132  0.6379 0.6301    0.9060   0.7406  0.7209  56.0s ✓
    2 frozen       0.8729  0.7529 0.7495    0.8418   0.7872  0.7748  54.0s ✓
  >>> Backbone unfrozen — trainable: 15.55M, LR → 1.0e-05
    3 finetune     0.7713  0.8173 0.8145    0.7715   0.8224  0.8122  62.5s ✓
    4 finetune     0.7102  0.8482 0.8460    0.7227   0.8408  0.8346  62.2s ✓
    5 finetune     0.6841  0.8649 0.8632    0.7252   0.8423  0.8318  62.4s ✓
    6 finetune     0.6596  0.8764 0.8753    0.7363   0.8467  0.8360  61.9s ✓
    7 finetune     0.6447  0.8837 0.8825    0.7322   0.8403  0.8304  61.8s
    8 finetune     0.6333  0.8916 0.8904    0.6873   0.8581  0.8501  62.1s ✓
    9 finetune     0.6209  0.8963 0.8952    0.7374   0.8398  0.8339  61.9s
   10 finetune     0.6113  0.9027 0.9019    0.6631

## 📈 Cell 6 — Research-Grade Graphs
The following plots are generated and saved as high-resolution PNG files
suitable for inclusion in the research paper:
 
1. **Training & Validation Loss curves** — per model
2. **Training & Validation Accuracy curves** — per model
3. **Confusion Matrix** — per model (normalised %)
4. **ROC Curves** — per model (one curve per class + macro AUC)
5. **Model Comparison Bar Chart** — Accuracy, Recall, F1, Specificity

In [8]:
SAVE_DIR = "/kaggle/working/plots"
os.makedirs(SAVE_DIR, exist_ok=True)
 
# ── Shared style ──────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi"       : 150,
    "savefig.dpi"      : 300,       # publication quality
    "font.family"      : "DejaVu Sans",
    "font.size"        : 11,
    "axes.titlesize"   : 13,
    "axes.labelsize"   : 11,
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "axes.grid"        : True,
    "grid.alpha"       : 0.3,
    "lines.linewidth"  : 2,
    "legend.framealpha": 0.9,
})
 
COLORS = {
    "resnet50": "#2563EB",   # blue
    "resnet18": "#16A34A",   # green
    "vgg16"   : "#DC2626",   # red
}
PHASE_COLOR = "#F59E0B"      # amber — freeze boundary line
 
SHORT = {"resnet50": "ResNet50", "resnet18": "ResNet18", "vgg16": "VGG16"}
 
# ── Custom confusion-matrix colormap (white → deep blue) ──
CM_CMAP = LinearSegmentedColormap.from_list(
    "cm_blue", ["#FFFFFF", "#DBEAFE", "#2563EB", "#1E3A8A"])
 
# ─────────────────────────────────────────────────────────
# PLOT 1 & 2 — Loss & Accuracy curves (one figure per model)
# ─────────────────────────────────────────────────────────
for backbone, r in all_results.items():
    hist   = r["history"]
    epochs = range(1, len(hist["train_loss"]) + 1)
    color  = COLORS[backbone]
 
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(f"{SHORT[backbone]} — Training Curves", fontsize=15, fontweight="bold")
 
    for ax, tr_key, vl_key, ylabel, title in [
        (axes[0], "train_loss", "val_loss",  "Loss",     "Loss Curve"),
        (axes[1], "train_acc",  "val_acc",   "Accuracy", "Accuracy Curve"),
    ]:
        ax.plot(epochs, hist[tr_key], color=color,     label="Train",      linestyle="--")
        ax.plot(epochs, hist[vl_key], color=color,     label="Validation", linestyle="-",
                linewidth=2.5)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.legend()
 
        # Mark freeze/finetune boundary
        if FREEZE_EPOCHS < EPOCHS:
            ax.axvline(x=FREEZE_EPOCHS + 0.5, color=PHASE_COLOR,
                       linestyle=":", linewidth=1.8, label=f"Unfreeze @ ep {FREEZE_EPOCHS+1}")
            ax.legend()
 
    plt.tight_layout()
    path = f"{SAVE_DIR}/{backbone}_loss_acc_curves.png"
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    print(f"Saved: {path}")
 
# ─────────────────────────────────────────────────────────
# PLOT 3 — Confusion Matrices (one per model)
# ─────────────────────────────────────────────────────────
SHORT_CLASS = ["Bact.", "COVID", "Normal", "TB", "Viral"]   # abbreviated for matrix
 
for backbone, r in all_results.items():
    C    = r["te_met"]["confusion"].astype(float)
    C_pct = C / C.sum(axis=1, keepdims=True) * 100   # row-normalised %
 
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(C_pct, cmap=CM_CMAP, vmin=0, vmax=100)
    plt.colorbar(im, ax=ax, label="Predicted %")
 
    ax.set_xticks(range(NUM_CLASSES))
    ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(SHORT_CLASS, rotation=30, ha="right")
    ax.set_yticklabels(SHORT_CLASS)
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")
    ax.set_title(f"{SHORT[backbone]} — Confusion Matrix (Test Set, %)",
                 fontweight="bold", pad=12)
 
    # Annotate cells
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            txt_color = "white" if C_pct[i, j] > 55 else "black"
            ax.text(j, i, f"{C_pct[i,j]:.1f}%\n({int(C[i,j])})",
                    ha="center", va="center",
                    fontsize=9, color=txt_color, fontweight="bold")
 
    plt.tight_layout()
    path = f"{SAVE_DIR}/{backbone}_confusion_matrix.png"
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    print(f"Saved: {path}")
 
# ─────────────────────────────────────────────────────────
# PLOT 4 — ROC Curves (one figure per model, one curve per class)
# ─────────────────────────────────────────────────────────
ROC_COLORS = ["#2563EB", "#16A34A", "#DC2626", "#9333EA", "#F59E0B"]
 
for backbone, r in all_results.items():
    y_true  = np.array(r["te_true"])
    y_probs = r["te_probs"]           # (N, C) numpy array
 
    # One-hot encode true labels for ROC computation
    y_onehot = np.zeros((len(y_true), NUM_CLASSES))
    for i, t in enumerate(y_true):
        y_onehot[i, t] = 1
 
    fig, ax = plt.subplots(figsize=(8, 7))
    macro_tpr, macro_fpr_base = [], np.linspace(0, 1, 200)
 
    for i, (cls_name, col) in enumerate(zip(CLASS_NAMES, ROC_COLORS)):
        fpr, tpr, _ = roc_curve(y_onehot[:, i], y_probs[:, i])
        roc_auc     = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=col, linewidth=2,
                label=f"{SHORT_CLASS[i]}  (AUC = {roc_auc:.3f})")
        # Interpolate for macro average
        macro_tpr.append(np.interp(macro_fpr_base, fpr, tpr))
 
    macro_tpr_mean = np.mean(macro_tpr, axis=0)
    macro_auc      = auc(macro_fpr_base, macro_tpr_mean)
    ax.plot(macro_fpr_base, macro_tpr_mean, color="black",
            linewidth=2.8, linestyle="--",
            label=f"Macro avg  (AUC = {macro_auc:.3f})")
 
    ax.plot([0, 1], [0, 1], "gray", linewidth=1, linestyle=":")
    ax.set_xlabel("False Positive Rate (1 − Specificity)")
    ax.set_ylabel("True Positive Rate (Sensitivity / Recall)")
    ax.set_title(f"{SHORT[backbone]} — ROC Curves (Test Set)",
                 fontweight="bold", pad=12)
    ax.legend(loc="lower right", fontsize=9)
    ax.set_xlim([-0.01, 1.01])
    ax.set_ylim([-0.01, 1.01])
 
    plt.tight_layout()
    path = f"{SAVE_DIR}/{backbone}_roc_curves.png"
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    print(f"Saved: {path}")
 
# ─────────────────────────────────────────────────────────
# PLOT 5 — Model Comparison Bar Chart (all 4 metrics)
# ─────────────────────────────────────────────────────────
metric_keys   = ["accuracy", "recall", "f1", "specificity"]
metric_labels = ["Accuracy", "Recall\n(Macro)", "F1 Score\n(Macro)", "Specificity\n(Macro)"]
n_metrics     = len(metric_keys)
n_models      = len(BACKBONES)
bar_width     = 0.22
x             = np.arange(n_metrics)
 
fig, ax = plt.subplots(figsize=(12, 6))
 
for idx, backbone in enumerate(BACKBONES):
    vals   = [all_results[backbone]["te_met"][k] * 100 for k in metric_keys]
    offset = (idx - n_models / 2 + 0.5) * bar_width
    bars   = ax.bar(x + offset, vals, bar_width,
                    label=SHORT[backbone],
                    color=COLORS[backbone], alpha=0.88,
                    edgecolor="white", linewidth=0.8)
    # Value labels on bars
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.4,
                f"{val:.1f}%", ha="center", va="bottom",
                fontsize=8.5, fontweight="bold",
                color=COLORS[backbone])
 
ax.set_xticks(x)
ax.set_xticklabels(metric_labels, fontsize=11)
ax.set_ylabel("Score (%)")
ax.set_title("Model Comparison — Test Set Performance",
             fontweight="bold", fontsize=14, pad=14)
ax.set_ylim([0, 108])
ax.legend(loc="upper right", fontsize=10)
ax.yaxis.grid(True, alpha=0.35)
ax.set_axisbelow(True)
 
plt.tight_layout()
path = f"{SAVE_DIR}/model_comparison_bar.png"
plt.savefig(path, bbox_inches="tight")
plt.close()
print(f"Saved: {path}")
 
# ─────────────────────────────────────────────────────────
# PLOT 6 — Combined Loss+Acc curves for all 3 models on one figure
# ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("All Models — Training & Validation Curves",
             fontsize=15, fontweight="bold")
 
for backbone, r in all_results.items():
    hist   = r["history"]
    epochs = range(1, len(hist["train_loss"]) + 1)
    color  = COLORS[backbone]
    label  = SHORT[backbone]
 
    axes[0].plot(epochs, hist["train_loss"], color=color, linestyle="--", alpha=0.6)
    axes[0].plot(epochs, hist["val_loss"],   color=color, linestyle="-",
                 label=label, linewidth=2.2)
 
    axes[1].plot(epochs, hist["train_acc"],  color=color, linestyle="--", alpha=0.6)
    axes[1].plot(epochs, hist["val_acc"],    color=color, linestyle="-",
                 label=label, linewidth=2.2)
 
for ax, ylabel, title in [
    (axes[0], "Loss",     "Loss Curves (solid=val, dashed=train)"),
    (axes[1], "Accuracy", "Accuracy Curves (solid=val, dashed=train)"),
]:
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    if FREEZE_EPOCHS < EPOCHS:
        ax.axvline(x=FREEZE_EPOCHS + 0.5, color=PHASE_COLOR,
                   linestyle=":", linewidth=1.5, alpha=0.8)
 
plt.tight_layout()
path = f"{SAVE_DIR}/all_models_curves.png"
plt.savefig(path, bbox_inches="tight")
plt.close()
print(f"Saved: {path}")
 
# ── List all saved plots ──────────────────────────────────
print(f"\n{'='*55}")
print(f"  All plots saved to: {SAVE_DIR}")
print(f"{'='*55}")
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(f"{SAVE_DIR}/{f}") / 1024
    print(f"  {f:<45} {size:.0f} KB")

Saved: /kaggle/working/plots/resnet50_loss_acc_curves.png
Saved: /kaggle/working/plots/resnet18_loss_acc_curves.png
Saved: /kaggle/working/plots/vgg16_loss_acc_curves.png
Saved: /kaggle/working/plots/resnet50_confusion_matrix.png
Saved: /kaggle/working/plots/resnet18_confusion_matrix.png
Saved: /kaggle/working/plots/vgg16_confusion_matrix.png
Saved: /kaggle/working/plots/resnet50_roc_curves.png
Saved: /kaggle/working/plots/resnet18_roc_curves.png
Saved: /kaggle/working/plots/vgg16_roc_curves.png
Saved: /kaggle/working/plots/model_comparison_bar.png
Saved: /kaggle/working/plots/all_models_curves.png

  All plots saved to: /kaggle/working/plots
  all_models_curves.png                         426 KB
  model_comparison_bar.png                      149 KB
  resnet18_confusion_matrix.png                 231 KB
  resnet18_loss_acc_curves.png                  243 KB
  resnet18_roc_curves.png                       236 KB
  resnet50_confusion_matrix.png                 228 KB
  resnet50_loss_acc

## 📤 Cell 7 — Extract Feature Vectors (512-d)
Extracts the 512-dimensional feature vectors from each trained model.   
These feed into the **Feature Fusion** module of the full multimodal pipeline.   
Each image → one 512-d vector. Saved as `.pt` files.

In [9]:
_, val_loader_feat = build_dataloaders(SUBSET_FRAC)
 
for backbone in BACKBONES:
    print(f"\nExtracting features: {backbone.upper()}")
 
    model = LungCNNClassifier(backbone, NUM_CLASSES, FEATURE_DIM,
                              pretrained=False).to(DEVICE)
    ckpt  = torch.load(f"/kaggle/working/{backbone}_best.pth", weights_only=False)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
 
    all_features, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in val_loader_feat:
            feats = model.extract_features(imgs.to(DEVICE))
            all_features.append(feats.cpu())
            all_labels.append(labels)
 
    all_features = torch.cat(all_features, dim=0)
    all_labels   = torch.cat(all_labels,   dim=0)
 
    feat_path = f"/kaggle/working/{backbone}_features.pt"
    torch.save({
        "features"   : all_features,
        "labels"     : all_labels,
        "class_names": CLASS_NAMES,
        "backbone"   : backbone,
    }, feat_path)
    print(f"  Feature shape : {tuple(all_features.shape)}")
    print(f"  Saved         → {feat_path}")
 
print("\n✓ All feature vectors saved.")
print("\nAll output files in /kaggle/working/:")
all_files = ([(f, "/kaggle/working") for f in os.listdir("/kaggle/working")
              if not os.path.isdir(f"/kaggle/working/{f}")
              and not f.endswith(".db")] + 
             [(f, SAVE_DIR) for f in os.listdir(SAVE_DIR)])
for fname, folder in sorted(all_files):
    size = os.path.getsize(f"{folder}/{fname}") / 1e6
    print(f"  {fname:<48} {size:.1f} MB")

  [TRAIN] Loading 6,054 images into RAM... done in 62.9s
  [VAL  ] Loading 2,016 images into RAM... done in 20.1s

Extracting features: RESNET50
  Feature shape : (2016, 512)
  Saved         → /kaggle/working/resnet50_features.pt

Extracting features: RESNET18
  Feature shape : (2016, 512)
  Saved         → /kaggle/working/resnet18_features.pt

Extracting features: VGG16
  Feature shape : (2016, 512)
  Saved         → /kaggle/working/vgg16_features.pt

✓ All feature vectors saved.

All output files in /kaggle/working/:
  __notebook__.ipynb                               0.1 MB
  all_models_curves.png                            0.4 MB
  model_comparison_bar.png                         0.2 MB
  resnet18_best.pth                                48.1 MB
  resnet18_confusion_matrix.png                    0.2 MB
  resnet18_features.pt                             4.1 MB
  resnet18_loss_acc_curves.png                     0.2 MB
  resnet18_roc_curves.png                          0.2 MB
  resnet50